<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/04_Statistical_Baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================================================
# NOTEBOOK 04 — STATISTICAL BASELINES
# SPP-GAN: A Privacy-Preserving Statistical–Machine Learning Framework for
# High-Fidelity Synthetic Tabular Data Generation
# ==================================================================================================
#
# PURPOSE
# -------
# Establish reproducible statistical baseline generators using the authoritative
# native TRAINING data persisted by Notebook 02.
#
# BASELINES
# ---------
# 1. Independent Marginal Sampling
# 2. Gaussian Copula
#
# DATA POLICY
# -----------
# - TRAIN split only is used for fitting.
# - VALIDATION and TEST are never used for fitting.
# - Notebook 02 preprocessing is NOT refitted.
# - Raw data are NOT reloaded.
#
# SCHEMA POLICY
# -------------
# Native generative schema:
#     preprocessing features + target
#
# Statistical model input:
#     generative columns
#
# Target:
#     generated as part of the synthetic generative table
#     NEVER used as a predictor
#
# Identifier / provenance:
#     excluded from synthetic generation
#
# RAM POLICY
# ----------
# - One dataset at a time.
# - One baseline at a time.
# - No dataset concatenation.
# - Persist artifacts immediately.
# - Explicit garbage collection.
#
# ==================================================================================================

print("=" * 100)
print("NOTEBOOK 04 — STATISTICAL BASELINES")
print("=" * 100)

NOTEBOOK_ID = "04"
NOTEBOOK_NAME = "Statistical Baselines"
NOTEBOOK_VERSION = "2.0"

print(f"Notebook       : {NOTEBOOK_ID} — {NOTEBOOK_NAME}")
print(f"Version        : {NOTEBOOK_VERSION}")
print("Purpose        : Reproducible statistical synthetic-data baselines")
print("Data policy    : TRAIN ONLY")
print("Schema policy  : Notebook 02 native generative schema")
print("RAM policy     : One dataset / one baseline at a time")

NOTEBOOK 04 — STATISTICAL BASELINES
Notebook       : 04 — Statistical Baselines
Version        : 2.0
Purpose        : Reproducible statistical synthetic-data baselines
Data policy    : TRAIN ONLY
Schema policy  : Notebook 02 native generative schema
RAM policy     : One dataset / one baseline at a time


In [2]:
# ==================================================================================================
# SECTION 2 — LOAD CONFIGURATION
# ==================================================================================================
#
# PURPOSE
# -------
# Load and validate the canonical Notebook 00 configuration for Notebook 04.
#
# METHODOLOGICAL POLICY
# ---------------------
# - Notebook 00 is the authoritative source for experiment configuration.
# - Configuration is loaded from the canonical project configuration directory.
# - No experimental parameter is silently replaced by a hard-coded value.
# - The configuration must define the canonical dataset registry and reproducibility seed.
# - Notebook 04 uses TRAIN data only for statistical-baseline fitting.
# - Validation and TEST splits are never used for fitting.
#
# NOTEBOOK 04 BASELINE POLICY
# ---------------------------
# Statistical baseline configuration is explicitly restricted to:
#
#   1. independent_marginal
#   2. gaussian_copula
#
# These method names are canonical identifiers used by subsequent sections.
#
# ==================================================================================================

print("=" * 100)
print("SECTION 2 — LOAD CONFIGURATION")
print("=" * 100)


# ==================================================================================================
# 2.1 — IMPORT REQUIRED LIBRARIES
# ==================================================================================================

from pathlib import Path
import json
import hashlib
import yaml
import pandas as pd
import numpy as np


# ==================================================================================================
# 2.2 — VERIFY GOOGLE DRIVE
# ==================================================================================================

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "Google Colab Drive interface could not be imported. "
        "Notebook 04 is designed for Google Colab."
    ) from exc


DRIVE_ROOT = Path(
    "/content/drive"
)

MYDRIVE_ROOT = (
    DRIVE_ROOT
    / "MyDrive"
)


if not MYDRIVE_ROOT.exists():

    print(
        "Google Drive is not mounted."
    )

    print(
        "Mounting Google Drive..."
    )

    drive.mount(
        "/content/drive"
    )


if not MYDRIVE_ROOT.exists():

    raise FileNotFoundError(
        "Google Drive mount failed.\n"
        f"Expected directory:\n{MYDRIVE_ROOT}"
    )


print(
    f"✓ Google Drive verified : {MYDRIVE_ROOT}"
)


# ==================================================================================================
# 2.3 — DEFINE CANONICAL PROJECT ROOT
# ==================================================================================================

PROJECT_NAME = (
    "SPP_GAN_Research"
)

PROJECT_ROOT = (
    MYDRIVE_ROOT
    / PROJECT_NAME
)


if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        "SPP-GAN project root was not found.\n"
        f"Expected:\n{PROJECT_ROOT}"
    )


if not PROJECT_ROOT.is_dir():

    raise NotADirectoryError(
        "SPP-GAN project root is not a directory.\n"
        f"Found:\n{PROJECT_ROOT}"
    )


print(
    f"✓ Project root verified : {PROJECT_ROOT}"
)


# ==================================================================================================
# 2.4 — DEFINE CANONICAL CONFIGURATION DIRECTORY
# ==================================================================================================

CONFIG_ROOT = (
    PROJECT_ROOT
    / "config"
)


if not CONFIG_ROOT.exists():

    raise FileNotFoundError(
        "Canonical configuration directory was not found.\n"
        f"Expected:\n{CONFIG_ROOT}"
    )


if not CONFIG_ROOT.is_dir():

    raise NotADirectoryError(
        "Canonical configuration path is not a directory.\n"
        f"Found:\n{CONFIG_ROOT}"
    )


print(
    f"✓ Configuration root    : {CONFIG_ROOT}"
)


# ==================================================================================================
# 2.5 — DISCOVER CANONICAL YAML CONFIGURATION
# ==================================================================================================
#
# Notebook 00 is expected to persist the master configuration under PROJECT_ROOT/config.
#
# We first check the canonical filename if present.
# If it is absent, we allow discovery only when exactly one plausible YAML configuration
# exists. Multiple candidates are treated as an integrity error rather than guessed.
#
# ==================================================================================================

CANONICAL_CONFIG_CANDIDATES = [

    CONFIG_ROOT
    / "master_config.yaml",

    CONFIG_ROOT
    / "master_config.yml",

    CONFIG_ROOT
    / "config.yaml",

    CONFIG_ROOT
    / "config.yml",

]


CONFIG_PATH = None


for candidate in CANONICAL_CONFIG_CANDIDATES:

    if candidate.is_file():

        CONFIG_PATH = candidate

        break


# --------------------------------------------------------------------------------------------------
# Controlled fallback discovery
# --------------------------------------------------------------------------------------------------

if CONFIG_PATH is None:

    discovered_yaml_files = sorted(
        list(
            CONFIG_ROOT.glob(
                "*.yaml"
            )
        )
        +
        list(
            CONFIG_ROOT.glob(
                "*.yml"
            )
        )
    )

    if len(discovered_yaml_files) == 1:

        CONFIG_PATH = (
            discovered_yaml_files[0]
        )

    elif len(discovered_yaml_files) == 0:

        raise FileNotFoundError(
            "No YAML configuration file was found in the canonical "
            f"configuration directory:\n{CONFIG_ROOT}\n\n"
            "Notebook 04 requires the persisted Notebook 00 configuration."
        )

    else:

        raise RuntimeError(
            "Multiple YAML configuration files were found in the canonical "
            "configuration directory and no unique master configuration "
            "could be identified.\n\n"
            f"Directory : {CONFIG_ROOT}\n"
            +
            "\n".join(
                f"  - {path.name}"
                for path in discovered_yaml_files
            )
        )


print(
    f"✓ Configuration file   : {CONFIG_PATH}"
)


# ==================================================================================================
# 2.6 — LOAD YAML CONFIGURATION
# ==================================================================================================

with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as file:

    CONFIG = yaml.safe_load(
        file
    )


if CONFIG is None:

    raise RuntimeError(
        "The canonical configuration file is empty."
    )


if not isinstance(
    CONFIG,
    dict
):

    raise TypeError(
        "The canonical Notebook 00 configuration must deserialize "
        "to a dictionary."
    )


print(
    "✓ YAML configuration loaded successfully."
)


# ==================================================================================================
# 2.7 — CONFIGURATION FILE FINGERPRINT
# ==================================================================================================

with open(
    CONFIG_PATH,
    "rb"
) as file:

    CONFIG_SHA256 = (
        hashlib.sha256(
            file.read()
        ).hexdigest()
    )


CONFIG_FILE_SIZE_BYTES = int(
    CONFIG_PATH.stat().st_size
)


print(
    f"✓ Configuration SHA-256 : {CONFIG_SHA256}"
)

print(
    f"✓ Configuration size    : "
    f"{CONFIG_FILE_SIZE_BYTES:,} bytes"
)


# ==================================================================================================
# 2.8 — VALIDATE TOP-LEVEL CONFIGURATION
# ==================================================================================================

if not CONFIG:

    raise RuntimeError(
        "Notebook 00 configuration contains no configuration entries."
    )


print(
    f"✓ Configuration entries : {len(CONFIG)}"
)


# ==================================================================================================
# 2.9 — EXTRACT MASTER SEED
# ==================================================================================================
#
# The master seed is the authoritative reproducibility seed.
#
# Notebook 04 does not create a new arbitrary master seed.
#
# ==================================================================================================

MASTER_SEED = (
    CONFIG.get(
        "master_seed"
    )
)


if MASTER_SEED is None:

    MASTER_SEED = (
        CONFIG.get(
            "seed"
        )
    )


if MASTER_SEED is None:

    raise RuntimeError(
        "Notebook 00 configuration does not define "
        "'master_seed' or 'seed'."
    )


try:

    MASTER_SEED = int(
        MASTER_SEED
    )

except (
    TypeError,
    ValueError
) as exc:

    raise TypeError(
        "The configured master seed must be an integer."
    ) from exc


if MASTER_SEED < 0:

    raise ValueError(
        f"MASTER_SEED must be non-negative. "
        f"Found: {MASTER_SEED}"
    )


print(
    f"✓ Master seed           : {MASTER_SEED}"
)


# ==================================================================================================
# 2.10 — EXTRACT CANONICAL DATASET REGISTRY
# ==================================================================================================

KNOWN_DATASET_REGISTRY = {
    "adult_income": {
        "target": "income",
    },
    "bank_marketing": {
        "target": "y",
    },
    "diabetes_130us": {
        "target": "readmitted",
    },
}


dataset_registry = (
    CONFIG.get(
        "datasets"
    )
)


if dataset_registry is None:

    dataset_registry = (
        CONFIG.get(
            "dataset_registry"
        )
    )


if dataset_registry is None:

    raise RuntimeError(
        "Notebook 00 configuration does not contain a canonical "
        "dataset registry under 'datasets' or 'dataset_registry'."
    )


if not isinstance(
    dataset_registry,
    dict
):

    raise TypeError(
        "The canonical dataset registry must be a dictionary."
    )


DATASET_IDS = list(
    KNOWN_DATASET_REGISTRY.keys()
)


missing_dataset_ids = [
    dataset_id
    for dataset_id in DATASET_IDS
    if dataset_id not in dataset_registry
]


if missing_dataset_ids:

    raise RuntimeError(
        "Canonical datasets are missing from Notebook 00 configuration:\n"
        +
        "\n".join(
            f"  - {dataset_id}"
            for dataset_id in missing_dataset_ids
        )
    )


# ==================================================================================================
# 2.11 — ESTABLISH TARGET REGISTRY
# ==================================================================================================

TARGET_COLUMNS = {
    dataset_id:
        KNOWN_DATASET_REGISTRY[
            dataset_id
        ][
            "target"
        ]
    for dataset_id in DATASET_IDS
}


# --------------------------------------------------------------------------------------------------
# Validate target definitions against configuration where available
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    configured_dataset = (
        dataset_registry[
            dataset_id
        ]
    )

    if isinstance(
        configured_dataset,
        dict
    ):

        configured_target = (
            configured_dataset.get(
                "target"
            )
        )

        if (
            configured_target is not None
            and
            str(
                configured_target
            )
            !=
            TARGET_COLUMNS[
                dataset_id
            ]
        ):

            raise RuntimeError(
                f"Target definition mismatch for '{dataset_id}'.\n"
                f"Expected : {TARGET_COLUMNS[dataset_id]}\n"
                f"Config   : {configured_target}"
            )


print()
print(
    "Canonical dataset registry:"
)

for dataset_id in DATASET_IDS:

    print(
        f"  {dataset_id:<20} | "
        f"target={TARGET_COLUMNS[dataset_id]}"
    )


# ==================================================================================================
# 2.12 — DEFINE IDENTIFIER REGISTRY
# ==================================================================================================
#
# Identifier policy is inherited from the frozen Notebook 02 schema.
# This registry is used only as an explicit Section 2 policy reference.
#
# ==================================================================================================

IDENTIFIER_COLUMNS = {

    "adult_income":
        [],

    "bank_marketing":
        [],

    "diabetes_130us":
        [
            "encounter_id",
            "patient_nbr",
        ],

}


# ==================================================================================================
# 2.13 — DEFINE EXPECTED SPLITS
# ==================================================================================================

EXPECTED_SPLITS = [
    "train",
    "validation",
    "test",
]


# ==================================================================================================
# 2.14 — LOAD RELEVANT EXPERIMENT CONFIGURATION
# ==================================================================================================

MISSINGNESS_CONFIG = (
    CONFIG.get(
        "missingness",
        {}
    )
)


if not isinstance(
    MISSINGNESS_CONFIG,
    dict
):

    raise TypeError(
        "The 'missingness' configuration must be a dictionary."
    )


# --------------------------------------------------------------------------------------------------
# Extract configured missingness mechanisms
# --------------------------------------------------------------------------------------------------

CONFIGURED_MISSINGNESS_MECHANISMS = (
    MISSINGNESS_CONFIG.get(
        "mechanisms"
    )
)


if CONFIGURED_MISSINGNESS_MECHANISMS is None:

    CONFIGURED_MISSINGNESS_MECHANISMS = (
        MISSINGNESS_CONFIG.get(
            "missingness_mechanisms"
        )
    )


if CONFIGURED_MISSINGNESS_MECHANISMS is not None:

    CONFIGURED_MISSINGNESS_MECHANISMS = [
        str(
            mechanism
        ).upper()
        for mechanism
        in CONFIGURED_MISSINGNESS_MECHANISMS
    ]


# --------------------------------------------------------------------------------------------------
# Extract configured missingness rates
# --------------------------------------------------------------------------------------------------

CONFIGURED_MISSINGNESS_RATES = (
    MISSINGNESS_CONFIG.get(
        "rates"
    )
)


if CONFIGURED_MISSINGNESS_RATES is None:

    CONFIGURED_MISSINGNESS_RATES = (
        MISSINGNESS_CONFIG.get(
            "missingness_rates"
        )
    )


if CONFIGURED_MISSINGNESS_RATES is not None:

    CONFIGURED_MISSINGNESS_RATES = [
        float(
            rate
        )
        for rate
        in CONFIGURED_MISSINGNESS_RATES
    ]


# --------------------------------------------------------------------------------------------------
# Extract configured repetitions
# --------------------------------------------------------------------------------------------------

CONFIGURED_REPETITIONS = (
    MISSINGNESS_CONFIG.get(
        "repetitions"
    )
)


if CONFIGURED_REPETITIONS is not None:

    CONFIGURED_REPETITIONS = int(
        CONFIGURED_REPETITIONS
    )


# ==================================================================================================
# 2.15 — DEFINE EXACT NOTEBOOK 04 BASELINE REGISTRY
# ==================================================================================================
#
# These are the only statistical baselines permitted in Notebook 04.
#
# Their mathematical definitions are implemented in Section 5.
#
# Section 2 only establishes the immutable method registry.
#
# ==================================================================================================

STATISTICAL_BASELINE_REGISTRY = {

    "independent_marginal": {

        "display_name":
            "Independent Marginal Sampling",

        "family":
            "statistical",

        "dependency_structure":
            "independent_marginals",

        "uses_cross_feature_dependency":
            False,

        "generates_target":
            True,

        "target_as_predictor":
            False,

        "requires_fitted_model":
            True,

        "random_state_required":
            True,

    },

    "gaussian_copula": {

        "display_name":
            "Gaussian Copula",

        "family":
            "statistical",

        "dependency_structure":
            "gaussian_copula",

        "uses_cross_feature_dependency":
            True,

        "generates_target":
            True,

        "target_as_predictor":
            False,

        "requires_fitted_model":
            True,

        "random_state_required":
            True,

    },

}


BASELINE_IDS = list(
    STATISTICAL_BASELINE_REGISTRY.keys()
)


if BASELINE_IDS != [
    "independent_marginal",
    "gaussian_copula",
]:

    raise RuntimeError(
        "Notebook 04 statistical baseline registry does not match "
        "the frozen baseline specification."
    )


# ==================================================================================================
# 2.16 — STATISTICAL BASELINE FITTING POLICY
# ==================================================================================================

STATISTICAL_FIT_POLICY = {

    "source_split":
        "train_only",

    "source_data":
        "Notebook_02_authoritative_native_training_data",

    "validation_used_for_fitting":
        False,

    "test_used_for_fitting":
        False,

    "raw_data_reloaded":
        False,

    "notebook_02_preprocessing_refitted":
        False,

    "encoded_data_used":
        False,

    "target_used_as_predictor":
        False,

    "target_generated":
        True,

    "identifier_columns_used":
        False,

    "provenance_used":
        False,

    "dataset_processing":
        "one_dataset_at_a_time",

    "baseline_processing":
        "one_baseline_at_a_time",

}


# ==================================================================================================
# 2.17 — CONFIGURATION VALIDATION
# ==================================================================================================

CONFIGURATION_CHECKS = {

    "configuration_is_dictionary":
        isinstance(
            CONFIG,
            dict
        ),

    "configuration_file_exists":
        CONFIG_PATH.is_file(),

    "master_seed_is_integer":
        isinstance(
            MASTER_SEED,
            int
        ),

    "master_seed_non_negative":
        MASTER_SEED >= 0,

    "dataset_registry_is_dictionary":
        isinstance(
            dataset_registry,
            dict
        ),

    "canonical_dataset_count":
        len(
            DATASET_IDS
        ) == 3,

    "adult_income_registered":
        "adult_income" in DATASET_IDS,

    "bank_marketing_registered":
        "bank_marketing" in DATASET_IDS,

    "diabetes_130us_registered":
        "diabetes_130us" in DATASET_IDS,

    "adult_income_target":
        TARGET_COLUMNS[
            "adult_income"
        ] == "income",

    "bank_marketing_target":
        TARGET_COLUMNS[
            "bank_marketing"
        ] == "y",

    "diabetes_130us_target":
        TARGET_COLUMNS[
            "diabetes_130us"
        ] == "readmitted",

    "expected_splits_complete":
        EXPECTED_SPLITS
        ==
        [
            "train",
            "validation",
            "test",
        ],

    "baseline_registry_complete":
        set(
            BASELINE_IDS
        )
        ==
        {
            "independent_marginal",
            "gaussian_copula",
        },

    "fit_policy_train_only":
        STATISTICAL_FIT_POLICY[
            "source_split"
        ]
        ==
        "train_only",

    "validation_excluded_from_fitting":
        STATISTICAL_FIT_POLICY[
            "validation_used_for_fitting"
        ]
        is False,

    "test_excluded_from_fitting":
        STATISTICAL_FIT_POLICY[
            "test_used_for_fitting"
        ]
        is False,

    "target_not_predictor":
        STATISTICAL_FIT_POLICY[
            "target_used_as_predictor"
        ]
        is False,

    "identifiers_excluded":
        STATISTICAL_FIT_POLICY[
            "identifier_columns_used"
        ]
        is False,

    "provenance_excluded":
        STATISTICAL_FIT_POLICY[
            "provenance_used"
        ]
        is False,

    "encoded_data_excluded":
        STATISTICAL_FIT_POLICY[
            "encoded_data_used"
        ]
        is False,

}


for check_name, passed in CONFIGURATION_CHECKS.items():

    print(
        f"  {'PASS' if passed else 'FAIL':<6} "
        f"{check_name}"
    )


CONFIGURATION_PASS = all(
    CONFIGURATION_CHECKS.values()
)


if not CONFIGURATION_PASS:

    failed_checks = [
        check_name
        for check_name, passed
        in CONFIGURATION_CHECKS.items()
        if not passed
    ]

    raise RuntimeError(
        "Notebook 04 Section 2 configuration validation failed:\n"
        +
        "\n".join(
            f"  - {check_name}"
            for check_name
            in failed_checks
        )
    )


# ==================================================================================================
# 2.18 — FINAL CONFIGURATION SUMMARY
# ==================================================================================================

print()
print("=" * 100)
print("SECTION 2 — CONFIGURATION SUMMARY")
print("=" * 100)

print(
    f"Configuration file       : {CONFIG_PATH}"
)

print(
    f"Configuration SHA-256    : {CONFIG_SHA256}"
)

print(
    f"Master seed              : {MASTER_SEED}"
)

print(
    f"Datasets                 : {len(DATASET_IDS)}"
)

print(
    f"Dataset IDs              : {DATASET_IDS}"
)

print(
    f"Expected splits          : {EXPECTED_SPLITS}"
)

print(
    f"Statistical baselines    : {len(BASELINE_IDS)}"
)

print(
    f"Baseline IDs             : {BASELINE_IDS}"
)

print(
    f"Fit source               : TRAIN ONLY"
)

print(
    f"Validation for fitting  : NO"
)

print(
    f"Test for fitting        : NO"
)

print(
    f"Target as predictor     : NO"
)

print(
    f"Identifiers used        : NO"
)

print(
    f"Provenance used         : NO"
)

print(
    f"Encoded data used       : NO"
)

print()
print(
    "✓ SECTION 2 CONFIGURATION STATUS : PASS"
)
print("=" * 100)

SECTION 2 — LOAD CONFIGURATION
Google Drive is not mounted.
Mounting Google Drive...
Mounted at /content/drive
✓ Google Drive verified : /content/drive/MyDrive
✓ Project root verified : /content/drive/MyDrive/SPP_GAN_Research
✓ Configuration root    : /content/drive/MyDrive/SPP_GAN_Research/config


FileNotFoundError: No YAML configuration file was found in the canonical configuration directory:
/content/drive/MyDrive/SPP_GAN_Research/config

Notebook 04 requires the persisted Notebook 00 configuration.

In [53]:
# ==================================================================================================
# 3. LOAD PROCESSED TRAINING DATA
# ==================================================================================================
#
# Notebook 04 MUST:
#   - use Notebook 02 native TRAIN data
#   - use the persisted Notebook 02 manifest
#   - preserve the Notebook 02 schema
#
# Notebook 04 MUST NOT:
#   - reload raw data
#   - reconstruct preprocessing
#   - refit preprocessing
#   - use encoded/scaled matrices
#   - use validation/test for fitting
#
# ==================================================================================================

print("=" * 100)
print("SECTION 3 — LOAD PROCESSED TRAINING DATA")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Canonical Notebook 02 locations
# --------------------------------------------------------------------------------------------------

CANONICAL_NB02_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_02"
)

CANONICAL_NATIVE_ROOT = (
    CANONICAL_NB02_ROOT
    / "native"
)

CANONICAL_PREPROCESSORS_ROOT = (
    CANONICAL_NB02_ROOT
    / "preprocessors"
)

CANONICAL_SCHEMA_ROOT = (
    CANONICAL_NB02_ROOT
    / "schemas"
)

CANONICAL_METADATA_ROOT = (
    CANONICAL_SCHEMA_ROOT
    / "metadata"
)

CANONICAL_NATIVE_MANIFEST = (
    CANONICAL_NATIVE_ROOT
    / "native_dataset_manifest.csv"
)

CANONICAL_PREPROCESSOR_MANIFEST = (
    CANONICAL_SCHEMA_ROOT
    / "preprocessor_manifest.csv"
)

# --------------------------------------------------------------------------------------------------
# Verify canonical Notebook 02 layer
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("NOTEBOOK 02 CANONICAL ARTIFACT VERIFICATION")
print("-" * 100)

required_nb02_paths = {
    "Notebook 02 root": CANONICAL_NB02_ROOT,
    "Native root": CANONICAL_NATIVE_ROOT,
    "Preprocessor root": CANONICAL_PREPROCESSORS_ROOT,
    "Schema root": CANONICAL_SCHEMA_ROOT,
    "Metadata root": CANONICAL_METADATA_ROOT,
    "Native manifest": CANONICAL_NATIVE_MANIFEST,
    "Preprocessor manifest": CANONICAL_PREPROCESSOR_MANIFEST,
}

for label, path in required_nb02_paths.items():

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{label:<25}: {path}"
    )

missing_nb02_paths = [
    str(path)
    for path in required_nb02_paths.values()
    if not path.exists()
]

if missing_nb02_paths:

    raise FileNotFoundError(
        "Required Notebook 02 persisted artifacts are incomplete:\n\n"
        + "\n".join(
            f"  - {path}"
            for path in missing_nb02_paths
        )
        + "\n\n"
        "Notebook 04 will NOT reconstruct Notebook 02."
    )

# --------------------------------------------------------------------------------------------------
# Load native manifest
# --------------------------------------------------------------------------------------------------

NATIVE_MANIFEST_DF = pd.read_csv(
    CANONICAL_NATIVE_MANIFEST,
    low_memory=False,
)

if NATIVE_MANIFEST_DF.empty:
    raise RuntimeError(
        "Notebook 02 native_dataset_manifest.csv is empty."
    )

print()
print(
    f"✓ Native manifest loaded : "
    f"{len(NATIVE_MANIFEST_DF)} rows"
)

# --------------------------------------------------------------------------------------------------
# Required manifest schema
# --------------------------------------------------------------------------------------------------

REQUIRED_NATIVE_MANIFEST_COLUMNS = {
    "dataset_id",
    "split",
    "relative_path",
    "absolute_path",
    "rows",
    "columns",
    "preprocessing_feature_columns",
    "generative_columns",
    "target_column",
    "provenance_column",
    "identifier_columns",
    "provenance_present",
    "target_present",
    "identifiers_excluded",
    "file_exists",
    "file_size_bytes",
    "sha256",
    "reload_validation",
    "status",
}

missing_manifest_columns = (
    REQUIRED_NATIVE_MANIFEST_COLUMNS
    - set(NATIVE_MANIFEST_DF.columns)
)

if missing_manifest_columns:

    raise RuntimeError(
        "Notebook 02 native manifest is missing required columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in sorted(
                missing_manifest_columns
            )
        )
    )

# --------------------------------------------------------------------------------------------------
# Normalize manifest values
# --------------------------------------------------------------------------------------------------

NATIVE_MANIFEST_DF["_dataset_id"] = (
    NATIVE_MANIFEST_DF["dataset_id"]
    .astype(str)
    .str.strip()
)

NATIVE_MANIFEST_DF["_split"] = (
    NATIVE_MANIFEST_DF["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# --------------------------------------------------------------------------------------------------
# Verify dataset coverage
# --------------------------------------------------------------------------------------------------

manifest_datasets = set(
    NATIVE_MANIFEST_DF["_dataset_id"]
)

if manifest_datasets != set(DATASET_IDS):

    raise RuntimeError(
        "Notebook 02 dataset coverage mismatch.\n"
        f"Expected: {DATASET_IDS}\n"
        f"Found   : {sorted(manifest_datasets)}"
    )

# --------------------------------------------------------------------------------------------------
# Verify exactly three splits per dataset
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    dataset_manifest = NATIVE_MANIFEST_DF[
        NATIVE_MANIFEST_DF["_dataset_id"] == dataset_id
    ]

    observed_splits = set(
        dataset_manifest["_split"]
    )

    if observed_splits != set(EXPECTED_SPLITS):

        raise RuntimeError(
            f"{dataset_id}: invalid split coverage.\n"
            f"Expected: {EXPECTED_SPLITS}\n"
            f"Found   : {sorted(observed_splits)}"
        )

    if len(dataset_manifest) != 3:

        raise RuntimeError(
            f"{dataset_id}: expected exactly 3 manifest rows."
        )

# --------------------------------------------------------------------------------------------------
# Select TRAIN rows only
# --------------------------------------------------------------------------------------------------

TRAIN_MANIFEST_DF = (
    NATIVE_MANIFEST_DF[
        NATIVE_MANIFEST_DF["_split"] == "train"
    ]
    .copy()
)

if len(TRAIN_MANIFEST_DF) != len(DATASET_IDS):

    raise RuntimeError(
        "Expected one TRAIN manifest record per dataset."
    )

# --------------------------------------------------------------------------------------------------
# Storage containers
# --------------------------------------------------------------------------------------------------

TRAINING_DATA = {}
TRAINING_ROWS = {}
TRAINING_COLUMNS = {}

TRAINING_FEATURE_COLUMNS = {}
TRAINING_GENERATIVE_COLUMNS = {}

TRAINING_TARGET_COLUMNS = {}
TRAINING_PROVENANCE_COLUMNS = {}
TRAINING_IDENTIFIER_COLUMNS = {}

TRAINING_DATA_PATHS = {}

# --------------------------------------------------------------------------------------------------
# SHA-256 helper
# --------------------------------------------------------------------------------------------------

def calculate_sha256(
    file_path,
    chunk_size=1024 * 1024,
):

    digest = hashlib.sha256()

    with open(file_path, "rb") as handle:

        while True:

            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()

# --------------------------------------------------------------------------------------------------
# Load each training dataset
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    manifest_row = TRAIN_MANIFEST_DF[
        TRAIN_MANIFEST_DF["_dataset_id"] == dataset_id
    ]

    if len(manifest_row) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one TRAIN manifest record."
        )

    manifest_row = manifest_row.iloc[0]

    training_path = Path(
        str(
            manifest_row["absolute_path"]
        ).strip()
    )

    if not training_path.exists():

        raise FileNotFoundError(
            f"{dataset_id}: TRAIN file not found:\n"
            f"{training_path}"
        )

    # ----------------------------------------------------------------------------------------------
    # File integrity
    # ----------------------------------------------------------------------------------------------

    expected_size = int(
        manifest_row["file_size_bytes"]
    )

    actual_size = training_path.stat().st_size

    if actual_size != expected_size:

        raise RuntimeError(
            f"{dataset_id}: file-size integrity failure."
        )

    expected_sha256 = str(
        manifest_row["sha256"]
    ).strip()

    actual_sha256 = calculate_sha256(
        training_path
    )

    if actual_sha256 != expected_sha256:

        raise RuntimeError(
            f"{dataset_id}: SHA-256 integrity failure."
        )

    # ----------------------------------------------------------------------------------------------
    # Load
    # ----------------------------------------------------------------------------------------------

    df = pd.read_csv(
        training_path,
        low_memory=False,
    )

    expected_rows = int(
        manifest_row["rows"]
    )

    if len(df) != expected_rows:

        raise RuntimeError(
            f"{dataset_id}: training row count mismatch.\n"
            f"Expected: {expected_rows}\n"
            f"Found   : {len(df)}"
        )

    expected_columns_count = int(
        manifest_row["columns"]
    )

    if len(df.columns) != expected_columns_count:

        raise RuntimeError(
            f"{dataset_id}: training column count mismatch."
        )

    # ----------------------------------------------------------------------------------------------
    # Parse manifest lists
    # ----------------------------------------------------------------------------------------------

    def parse_manifest_list(value):

        if isinstance(value, list):
            return value

        text = str(value).strip()

        try:
            parsed = json.loads(text)

        except Exception:

            parsed = ast.literal_eval(text)

        if not isinstance(parsed, list):

            raise ValueError(
                f"Expected list, received: {type(parsed)}"
            )

        return [
            str(item).strip()
            for item in parsed
        ]

    feature_columns = parse_manifest_list(
        manifest_row[
            "preprocessing_feature_columns"
        ]
    )

    generative_columns = parse_manifest_list(
        manifest_row[
            "generative_columns"
        ]
    )

    identifier_columns = parse_manifest_list(
        manifest_row[
            "identifier_columns"
        ]
    )

    target_column = str(
        manifest_row["target_column"]
    ).strip()

    provenance_column = str(
        manifest_row["provenance_column"]
    ).strip()

    # ----------------------------------------------------------------------------------------------
    # Store canonical objects
    # ----------------------------------------------------------------------------------------------

    TRAINING_DATA[dataset_id] = df

    TRAINING_ROWS[dataset_id] = len(df)

    TRAINING_COLUMNS[dataset_id] = list(
        df.columns
    )

    TRAINING_FEATURE_COLUMNS[dataset_id] = (
        feature_columns
    )

    TRAINING_GENERATIVE_COLUMNS[dataset_id] = (
        generative_columns
    )

    TRAINING_TARGET_COLUMNS[dataset_id] = (
        target_column
    )

    TRAINING_PROVENANCE_COLUMNS[dataset_id] = (
        provenance_column
    )

    TRAINING_IDENTIFIER_COLUMNS[dataset_id] = (
        identifier_columns
    )

    TRAINING_DATA_PATHS[dataset_id] = (
        training_path
    )

    print(
        f"✓ {dataset_id:<20} | "
        f"rows={len(df):>8,} | "
        f"native_cols={len(df.columns):>3} | "
        f"features={len(feature_columns):>3} | "
        f"generative={len(generative_columns):>3} | "
        f"target={target_column}"
    )

print()
print("✓ Notebook 02 native TRAIN data loaded.")
print("✓ Manifest-authoritative paths verified.")
print("✓ File-size integrity verified.")
print("✓ SHA-256 integrity verified.")

SECTION 3 — LOAD PROCESSED TRAINING DATA

----------------------------------------------------------------------------------------------------
NOTEBOOK 02 CANONICAL ARTIFACT VERIFICATION
----------------------------------------------------------------------------------------------------
✗ Notebook 02 root         : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
✗ Native root              : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native
✗ Preprocessor root        : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/preprocessors
✗ Schema root              : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas
✗ Metadata root            : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/metadata
✗ Native manifest          : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native/native_dataset_manifest.csv
✗ Preprocessor manifest    : /content/drive/MyDrive/SPP

FileNotFoundError: Required Notebook 02 persisted artifacts are incomplete:

  - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
  - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native
  - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/preprocessors
  - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas
  - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/metadata
  - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native/native_dataset_manifest.csv
  - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/preprocessor_manifest.csv

Notebook 04 will NOT reconstruct Notebook 02.

In [12]:
# ==================================================================================================
# 4. VALIDATE INPUT SCHEMA
# ==================================================================================================

print("=" * 100)
print("SECTION 4 — VALIDATE INPUT SCHEMA")
print("=" * 100)

INPUT_SCHEMA_SUMMARY = {}

for dataset_id in DATASET_IDS:

    df = TRAINING_DATA[dataset_id]

    features = TRAINING_FEATURE_COLUMNS[
        dataset_id
    ]

    generative_columns = TRAINING_GENERATIVE_COLUMNS[
        dataset_id
    ]

    target = TRAINING_TARGET_COLUMNS[
        dataset_id
    ]

    provenance = TRAINING_PROVENANCE_COLUMNS[
        dataset_id
    ]

    identifiers = TRAINING_IDENTIFIER_COLUMNS[
        dataset_id
    ]

    # ----------------------------------------------------------------------------------------------
    # Basic existence
    # ----------------------------------------------------------------------------------------------

    if not set(features).issubset(df.columns):

        raise RuntimeError(
            f"{dataset_id}: preprocessing feature columns missing."
        )

    if not set(generative_columns).issubset(df.columns):

        raise RuntimeError(
            f"{dataset_id}: generative columns missing."
        )

    if target not in df.columns:

        raise RuntimeError(
            f"{dataset_id}: target column missing."
        )

    if provenance not in df.columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column missing."
        )

    # ----------------------------------------------------------------------------------------------
    # Frozen Notebook 02 policy
    # ----------------------------------------------------------------------------------------------

    if target in features:

        raise RuntimeError(
            f"{dataset_id}: TARGET LEAKAGE — target appears in features."
        )

    if provenance in features:

        raise RuntimeError(
            f"{dataset_id}: PROVENANCE LEAKAGE."
        )

    identifier_overlap = (
        set(features)
        .intersection(identifiers)
    )

    if identifier_overlap:

        raise RuntimeError(
            f"{dataset_id}: IDENTIFIER LEAKAGE:\n"
            f"{sorted(identifier_overlap)}"
        )

    # ----------------------------------------------------------------------------------------------
    # Generative schema
    # ----------------------------------------------------------------------------------------------

    if target not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target is not retained in generative schema."
        )

    if provenance in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance appears in generative schema."
        )

    generative_identifier_overlap = (
        set(generative_columns)
        .intersection(identifiers)
    )

    if generative_identifier_overlap:

        raise RuntimeError(
            f"{dataset_id}: identifiers appear in generative schema:\n"
            f"{sorted(generative_identifier_overlap)}"
        )

    # ----------------------------------------------------------------------------------------------
    # Expected target registry
    # ----------------------------------------------------------------------------------------------

    if target != TARGET_COLUMNS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: target registry mismatch."
        )

    INPUT_SCHEMA_SUMMARY[dataset_id] = {
        "training_rows": len(df),
        "native_columns": len(df.columns),
        "feature_columns": len(features),
        "generative_columns": len(generative_columns),
        "target_column": target,
        "provenance_column": provenance,
        "identifier_columns": identifiers,
    }

    print(
        f"✓ {dataset_id:<20} | "
        f"features={len(features):>3} | "
        f"generative={len(generative_columns):>3} | "
        f"target={target} | "
        f"target/provenance/ID separation PASS"
    )

print()
print("✓ SECTION 4 — INPUT SCHEMA : PASS")

SECTION 4 — VALIDATE INPUT SCHEMA


NameError: name 'DATASET_IDS' is not defined

In [13]:
# ==================================================================================================
# 5. DEFINE EXACT STATISTICAL BASELINE METHODS
# ==================================================================================================

print("=" * 100)
print("SECTION 5 — DEFINE EXACT STATISTICAL BASELINE METHODS")
print("=" * 100)

BASELINE_DEFINITIONS = {

    "independent_marginal": {

        "name": "Independent Marginal Sampling",

        "family": "Statistical",

        "principle": (
            "Estimate the empirical marginal distribution of each "
            "generative variable independently and sample each "
            "variable independently."
        ),

        "dependency_model": "None",

        "privacy_guarantee": False,
    },

    "gaussian_copula": {

        "name": "Gaussian Copula",

        "family": "Statistical",

        "principle": (
            "Model individual variable distributions together with "
            "their dependence structure through a Gaussian copula."
        ),

        "dependency_model": "Gaussian copula",

        "privacy_guarantee": False,
    },
}

for baseline_name in BASELINE_METHODS:

    definition = BASELINE_DEFINITIONS[
        baseline_name
    ]

    print()
    print(
        f"{baseline_name}"
    )
    print(
        f"  Name        : {definition['name']}"
    )
    print(
        f"  Family      : {definition['family']}"
    )
    print(
        f"  Dependency  : {definition['dependency_model']}"
    )
    print(
        f"  DP guarantee: {definition['privacy_guarantee']}"
    )

print()
print("✓ Exact baseline registry frozen.")

SECTION 5 — DEFINE EXACT STATISTICAL BASELINE METHODS

Registered statistical baselines:
✓ independent_marginal      : Independent Marginal Sampling
✓ gaussian_copula           : Gaussian Copula

✓ SECTION 5 PASS


In [15]:
# ==================================================================================================
# 6. CONFIGURE STATISTICAL BASELINES
# ==================================================================================================

print("=" * 100)
print("SECTION 6 — CONFIGURE STATISTICAL BASELINES")
print("=" * 100)

BASELINE_CONFIG = {

    "independent_marginal": {

        "sampling": "empirical_distribution",

        "replacement": True,

        "sample_size": "training_rows",

        "fit_data": "native_train_only",
    },

    "gaussian_copula": {

        "default_distribution": "norm",

        "enforce_min_max_values": True,

        "enforce_rounding": True,

        "sample_size": "training_rows",

        "fit_data": "native_train_only",
    },
}

print(
    json.dumps(
        BASELINE_CONFIG,
        indent=2,
    )
)

print()
print("✓ Baseline configuration frozen.")

SECTION 6 — CONFIGURE STATISTICAL BASELINES
{
  "baselines": {
    "independent_marginal": {
      "enabled": true,
      "random_state": 2025,
      "sampling_mode": "empirical"
    },
    "gaussian_copula": {
      "enabled": true,
      "random_state": 2025,
      "enforce_minimum_probability": 1e-08,
      "correlation_regularization": 1e-06
    }
  },
  "generation": {
    "sample_size_policy": "MATCH_TRAINING_ROWS",
    "include_target": true,
    "include_identifiers": false,
    "include_provenance": false,
    "use_validation": false,
    "use_test": false,
    "use_synthetic_for_fitting": false
  },
  "resource": {
    "process_one_dataset_at_a_time": true,
    "process_one_method_at_a_time": true,
    "persist_immediately": true,
    "delete_intermediate_objects": true
  },
  "validation": {
    "exact_schema": true,
    "exact_column_order": true,
    "exact_sample_size": true,
    "no_identifier_columns": true,
    "no_provenance_column": true,
    "target_required": true


In [17]:
# ==================================================================================================
# 7. SET REPRODUCIBLE SEEDS
# ==================================================================================================

print("=" * 100)
print("SECTION 7 — SET REPRODUCIBLE SEEDS")
print("=" * 100)

def seed_everything(seed):

    random.seed(seed)

    np.random.seed(seed)

    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)


def get_baseline_seed(
    dataset_index,
    baseline_index,
):

    return (
        MASTER_SEED
        + ((dataset_index + 1) * 1000)
        + ((baseline_index + 1) * 100)
    )


seed_everything(
    MASTER_SEED
)

print(
    f"✓ MASTER_SEED = {MASTER_SEED}"
)

print(
    "✓ Python random seeded"
)

print(
    "✓ NumPy random seeded"
)

print(
    "✓ Dataset/baseline deterministic seed policy configured"
)

SECTION 7 — SET REPRODUCIBLE SEEDS
✓ Python seed : 2025
✓ NumPy seed  : 2025
✓ PyTorch seed: 2025
✓ CUDA available: True

✓ SECTION 7 PASS


In [19]:
# ==================================================================================================
# 8. FIT BASELINE DISTRIBUTIONS
# ==================================================================================================

print("=" * 100)
print("SECTION 8 — FIT BASELINE DISTRIBUTIONS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Install SDV only if required
# --------------------------------------------------------------------------------------------------

try:

    import sdv
    from sdv.metadata import Metadata
    from sdv.single_table import GaussianCopulaSynthesizer

except ImportError:

    print("SDV not installed. Installing...")

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "sdv",
        ]
    )

    import sdv
    from sdv.metadata import Metadata
    from sdv.single_table import GaussianCopulaSynthesizer


print(
    f"✓ SDV version : "
    f"{getattr(sdv, '__version__', 'unknown')}"
)

# --------------------------------------------------------------------------------------------------
# Independent marginal fitter
# --------------------------------------------------------------------------------------------------

def fit_independent_marginal(
    dataframe,
    columns,
):

    fitted = {}

    for column in columns:

        series = dataframe[column]

        # Preserve missing values as an empirical state.
        values = (
            series.astype(object)
            .where(
                series.notna(),
                "__NB04_MISSING__",
            )
        )

        frequencies = (
            values
            .value_counts(
                normalize=True,
                dropna=False,
            )
        )

        fitted[column] = {

            "values": frequencies.index.tolist(),

            "probabilities": (
                frequencies.values
                .astype(float)
                .tolist()
            ),

            "dtype": str(
                series.dtype
            ),

            "n_unique": int(
                series.nunique(
                    dropna=False
                )
            ),
        }

    return fitted


# --------------------------------------------------------------------------------------------------
# Model containers
# --------------------------------------------------------------------------------------------------

FITTED_BASELINE_MODELS = {}

FIT_RUNTIME_RECORDS = []

# --------------------------------------------------------------------------------------------------
# Fit one dataset at a time
# --------------------------------------------------------------------------------------------------

for dataset_index, dataset_id in enumerate(
    DATASET_IDS
):

    print()
    print("-" * 100)
    print(
        f"FITTING — {dataset_id}"
    )
    print("-" * 100)

    FITTED_BASELINE_MODELS[
        dataset_id
    ] = {}

    train_df = TRAINING_DATA[
        dataset_id
    ]

    generative_columns = (
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    for baseline_index, baseline_name in enumerate(
        BASELINE_METHODS
    ):

        seed = get_baseline_seed(
            dataset_index,
            baseline_index,
        )

        seed_everything(seed)

        start_time = time.perf_counter()

        # ------------------------------------------------------------------------------------------
        # Independent Marginal
        # ------------------------------------------------------------------------------------------

        if baseline_name == "independent_marginal":

            model = fit_independent_marginal(
                dataframe=train_df[
                    generative_columns
                ],
                columns=generative_columns,
            )

            FITTED_BASELINE_MODELS[
                dataset_id
            ][baseline_name] = model

        # ------------------------------------------------------------------------------------------
        # Gaussian Copula
        # ------------------------------------------------------------------------------------------

        elif baseline_name == "gaussian_copula":

            model_input = (
                train_df[
                    generative_columns
                ]
                .copy()
            )

            metadata = (
                Metadata.detect_from_dataframe(
                    data=model_input,
                    table_name=dataset_id,
                    infer_keys=None,
                )
            )

            metadata.validate()

            synthesizer = (
                GaussianCopulaSynthesizer(
                    metadata,
                    enforce_min_max_values=True,
                    enforce_rounding=True,
                    default_distribution="norm",
                )
            )

            synthesizer.fit(
                model_input
            )

            FITTED_BASELINE_MODELS[
                dataset_id
            ][baseline_name] = {

                "synthesizer": synthesizer,

                "metadata": metadata,
            }

        else:

            raise ValueError(
                f"Unknown baseline: {baseline_name}"
            )

        elapsed = (
            time.perf_counter()
            - start_time
        )

        FIT_RUNTIME_RECORDS.append(
            {
                "dataset_id": dataset_id,
                "baseline": baseline_name,
                "seed": seed,
                "fit_runtime_seconds": elapsed,
            }
        )

        print(
            f"✓ {baseline_name:<24} | "
            f"fit_runtime={elapsed:.3f}s"
        )

        gc.collect()

print()
print("✓ All statistical baseline distributions fitted.")

SECTION 8 — FIT BASELINE DISTRIBUTIONS


NameError: name 'DATASET_IDS' is not defined

In [20]:
# ==================================================================================================
# 9. GENERATE SYNTHETIC DATA
# ==================================================================================================

print("=" * 100)
print("SECTION 9 — GENERATE SYNTHETIC DATA")
print("=" * 100)

SYNTHETIC_DATA = {}

GENERATION_RUNTIME_RECORDS = []

for dataset_index, dataset_id in enumerate(
    DATASET_IDS
):

    print()
    print("-" * 100)
    print(
        f"GENERATING — {dataset_id}"
    )
    print("-" * 100)

    training_rows = TRAINING_ROWS[
        dataset_id
    ]

    generative_columns = (
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    SYNTHETIC_DATA[
        dataset_id
    ] = {}

    for baseline_index, baseline_name in enumerate(
        BASELINE_METHODS
    ):

        seed = get_baseline_seed(
            dataset_index,
            baseline_index,
        )

        seed_everything(seed)

        start_time = time.perf_counter()

        # ------------------------------------------------------------------------------------------
        # Independent Marginal
        # ------------------------------------------------------------------------------------------

        if baseline_name == "independent_marginal":

            model = (
                FITTED_BASELINE_MODELS[
                    dataset_id
                ][baseline_name]
            )

            rng = np.random.default_rng(
                seed
            )

            synthetic_columns = {}

            for column in generative_columns:

                values = np.asarray(
                    model[column]["values"],
                    dtype=object,
                )

                probabilities = np.asarray(
                    model[column]["probabilities"],
                    dtype=float,
                )

                probabilities = (
                    probabilities
                    / probabilities.sum()
                )

                sampled = rng.choice(
                    values,
                    size=training_rows,
                    replace=True,
                    p=probabilities,
                )

                sampled_series = (
                    pd.Series(
                        sampled,
                        dtype=object,
                    )
                )

                sampled_series = (
                    sampled_series
                    .replace(
                        "__NB04_MISSING__",
                        np.nan,
                    )
                )

                synthetic_columns[
                    column
                ] = sampled_series

            synthetic_df = pd.DataFrame(
                synthetic_columns,
                columns=generative_columns,
            )

        # ------------------------------------------------------------------------------------------
        # Gaussian Copula
        # ------------------------------------------------------------------------------------------

        elif baseline_name == "gaussian_copula":

            synthesizer = (
                FITTED_BASELINE_MODELS[
                    dataset_id
                ][baseline_name][
                    "synthesizer"
                ]
            )

            synthetic_df = (
                synthesizer.sample(
                    num_rows=training_rows
                )
            )

            synthetic_df = (
                synthetic_df[
                    generative_columns
                ]
                .copy()
            )

        else:

            raise ValueError(
                f"Unknown baseline: {baseline_name}"
            )

        elapsed = (
            time.perf_counter()
            - start_time
        )

        SYNTHETIC_DATA[
            dataset_id
        ][baseline_name] = synthetic_df

        GENERATION_RUNTIME_RECORDS.append(
            {
                "dataset_id": dataset_id,
                "baseline": baseline_name,
                "seed": seed,
                "rows_requested": training_rows,
                "rows_generated": len(synthetic_df),
                "generation_runtime_seconds": elapsed,
            }
        )

        print(
            f"✓ {baseline_name:<24} | "
            f"rows={len(synthetic_df):>8,} | "
            f"runtime={elapsed:.3f}s"
        )

        gc.collect()

print()
print("✓ Synthetic datasets generated.")

SECTION 9 — GENERATE SYNTHETIC DATA


NameError: name 'DATASET_IDS' is not defined

In [21]:
# ==================================================================================================
# 10. VALIDATE SYNTHETIC SCHEMA
# ==================================================================================================

print("=" * 100)
print("SECTION 10 — VALIDATE SYNTHETIC SCHEMA")
print("=" * 100)

SYNTHETIC_SCHEMA_VALIDATION = []

for dataset_id in DATASET_IDS:

    expected_columns = (
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    target = (
        TRAINING_TARGET_COLUMNS[
            dataset_id
        ]
    )

    provenance = (
        TRAINING_PROVENANCE_COLUMNS[
            dataset_id
        ]
    )

    identifiers = (
        TRAINING_IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )

    for baseline_name in BASELINE_METHODS:

        synthetic_df = (
            SYNTHETIC_DATA[
                dataset_id
            ][baseline_name]
        )

        # ------------------------------------------------------------------------------------------
        # Exact column order
        # ------------------------------------------------------------------------------------------

        if list(
            synthetic_df.columns
        ) != list(
            expected_columns
        ):

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "synthetic column schema mismatch."
            )

        # ------------------------------------------------------------------------------------------
        # Target retained
        # ------------------------------------------------------------------------------------------

        if target not in synthetic_df.columns:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "target column missing."
            )

        # ------------------------------------------------------------------------------------------
        # Provenance excluded
        # ------------------------------------------------------------------------------------------

        if provenance in synthetic_df.columns:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "provenance leakage detected."
            )

        # ------------------------------------------------------------------------------------------
        # Identifiers excluded
        # ------------------------------------------------------------------------------------------

        identifier_leakage = (
            set(identifiers)
            .intersection(
                synthetic_df.columns
            )
        )

        if identifier_leakage:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                f"identifier leakage: "
                f"{sorted(identifier_leakage)}"
            )

        SYNTHETIC_SCHEMA_VALIDATION.append(
            {
                "dataset_id": dataset_id,
                "baseline": baseline_name,
                "schema_pass": True,
                "columns": len(
                    synthetic_df.columns
                ),
                "target_present": True,
                "provenance_excluded": True,
                "identifiers_excluded": True,
            }
        )

        print(
            f"✓ {dataset_id:<20} | "
            f"{baseline_name:<24} | "
            f"schema PASS"
        )

SYNTHETIC_SCHEMA_VALIDATION_DF = pd.DataFrame(
    SYNTHETIC_SCHEMA_VALIDATION
)

print()
print("✓ Synthetic schema validation : PASS")

SECTION 10 — VALIDATE SYNTHETIC SCHEMA

✓ SECTION 10 PASS


In [22]:
# ==================================================================================================
# 11. VALIDATE SAMPLE SIZE
# ==================================================================================================

print("=" * 100)
print("SECTION 11 — VALIDATE SAMPLE SIZE")
print("=" * 100)

for dataset_id in DATASET_IDS:

    expected_rows = TRAINING_ROWS[
        dataset_id
    ]

    for baseline_name in BASELINE_METHODS:

        actual_rows = len(
            SYNTHETIC_DATA[
                dataset_id
            ][baseline_name]
        )

        if actual_rows != expected_rows:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                f"sample-size mismatch.\n"
                f"Expected: {expected_rows}\n"
                f"Found   : {actual_rows}"
            )

        print(
            f"✓ {dataset_id:<20} | "
            f"{baseline_name:<24} | "
            f"{actual_rows:>8,} rows PASS"
        )

print()
print("✓ Synthetic sample-size validation : PASS")

SECTION 11 — VALIDATE SAMPLE SIZE

✓ SECTION 11 PASS


In [23]:
# ==================================================================================================
# 12. RECORD RUNTIME
# ==================================================================================================

print("=" * 100)
print("SECTION 12 — RECORD RUNTIME")
print("=" * 100)

FIT_RUNTIME_DF = pd.DataFrame(
    FIT_RUNTIME_RECORDS
)

GENERATION_RUNTIME_DF = pd.DataFrame(
    GENERATION_RUNTIME_RECORDS
)

RUNTIME_DF = (
    FIT_RUNTIME_DF
    .merge(
        GENERATION_RUNTIME_DF,
        on=[
            "dataset_id",
            "baseline",
            "seed",
        ],
        how="inner",
    )
)

RUNTIME_DF[
    "total_runtime_seconds"
] = (
    RUNTIME_DF[
        "fit_runtime_seconds"
    ]
    +
    RUNTIME_DF[
        "generation_runtime_seconds"
    ]
)

RUNTIME_DF = RUNTIME_DF[
    [
        "dataset_id",
        "baseline",
        "seed",
        "rows_requested",
        "rows_generated",
        "fit_runtime_seconds",
        "generation_runtime_seconds",
        "total_runtime_seconds",
    ]
]

print(
    RUNTIME_DF.to_string(
        index=False
    )
)

print()
print("✓ Runtime records created.")

SECTION 12 — RECORD RUNTIME


KeyError: "None of [Index(['dataset_id', 'method_id', 'fit_runtime_seconds'], dtype='object')] are in the [columns]"

In [24]:
# ==================================================================================================
# 13. SAVE SYNTHETIC DATA
# ==================================================================================================

print("=" * 100)
print("SECTION 13 — SAVE SYNTHETIC DATA")
print("=" * 100)

NB04_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_04"
)

NB04_SYNTHETIC_ROOT = (
    NB04_ROOT
    / "synthetic"
)

NB04_MODEL_ROOT = (
    NB04_ROOT
    / "models"
)

NB04_METADATA_ROOT = (
    NB04_ROOT
    / "metadata"
)

NB04_MANIFEST_ROOT = (
    NB04_ROOT
    / "manifests"
)

NB04_VALIDATION_ROOT = (
    NB04_ROOT
    / "validation"
)

NB04_SCHEMA_ROOT = (
    NB04_ROOT
    / "schemas"
)

for directory in [
    NB04_ROOT,
    NB04_SYNTHETIC_ROOT,
    NB04_MODEL_ROOT,
    NB04_METADATA_ROOT,
    NB04_MANIFEST_ROOT,
    NB04_VALIDATION_ROOT,
    NB04_SCHEMA_ROOT,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

SYNTHETIC_ARTIFACT_RECORDS = []

for dataset_id in DATASET_IDS:

    dataset_root = (
        NB04_SYNTHETIC_ROOT
        / dataset_id
    )

    dataset_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    for baseline_name in BASELINE_METHODS:

        synthetic_df = (
            SYNTHETIC_DATA[
                dataset_id
            ][baseline_name]
        )

        output_path = (
            dataset_root
            / f"{baseline_name}.csv"
        )

        synthetic_df.to_csv(
            output_path,
            index=False,
        )

        file_hash = calculate_sha256(
            output_path
        )

        SYNTHETIC_ARTIFACT_RECORDS.append(
            {
                "dataset_id": dataset_id,
                "baseline": baseline_name,
                "artifact_type": "synthetic_data",
                "path": str(output_path),
                "relative_path": str(
                    output_path.relative_to(
                        NB04_ROOT
                    )
                ),
                "rows": len(synthetic_df),
                "columns": len(
                    synthetic_df.columns
                ),
                "file_size_bytes": (
                    output_path.stat().st_size
                ),
                "sha256": file_hash,
            }
        )

        print(
            f"✓ {dataset_id:<20} | "
            f"{baseline_name:<24} | "
            f"{output_path}"
        )

SYNTHETIC_ARTIFACT_DF = pd.DataFrame(
    SYNTHETIC_ARTIFACT_RECORDS
)

print()
print(
    f"✓ Synthetic datasets saved : "
    f"{len(SYNTHETIC_ARTIFACT_DF)}"
)

SECTION 13 — SAVE SYNTHETIC DATA

✓ SECTION 13 PASS


In [25]:
# ==================================================================================================
# 14. SAVE BASELINE METADATA
# ==================================================================================================

print("=" * 100)
print("SECTION 14 — SAVE BASELINE METADATA")
print("=" * 100)

BASELINE_METADATA_RECORDS = []

for dataset_id in DATASET_IDS:

    train_df = TRAINING_DATA[
        dataset_id
    ]

    generative_columns = (
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    feature_columns = (
        TRAINING_FEATURE_COLUMNS[
            dataset_id
        ]
    )

    target = (
        TRAINING_TARGET_COLUMNS[
            dataset_id
        ]
    )

    provenance = (
        TRAINING_PROVENANCE_COLUMNS[
            dataset_id
        ]
    )

    identifiers = (
        TRAINING_IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )

    numeric_columns = [
        column
        for column in generative_columns
        if pd.api.types.is_numeric_dtype(
            train_df[column]
        )
    ]

    categorical_columns = [
        column
        for column in generative_columns
        if column not in numeric_columns
    ]

    for baseline_index, baseline_name in enumerate(
        BASELINE_METHODS
    ):

        seed = get_baseline_seed(
            DATASET_IDS.index(dataset_id),
            baseline_index,
        )

        metadata = {

            "notebook": {
                "id": NOTEBOOK_ID,
                "name": NOTEBOOK_NAME,
                "version": NOTEBOOK_VERSION,
            },

            "dataset_id": dataset_id,

            "baseline": {
                "id": baseline_name,
                "name": BASELINE_DEFINITIONS[
                    baseline_name
                ]["name"],
                "family": BASELINE_DEFINITIONS[
                    baseline_name
                ]["family"],
                "definition": BASELINE_DEFINITIONS[
                    baseline_name
                ]["principle"],
                "dependency_model": BASELINE_DEFINITIONS[
                    baseline_name
                ]["dependency_model"],
                "formal_dp": False,
            },

            "data_source": {
                "notebook": "02",
                "layer": "native",
                "fit_split": "train",
                "validation_used": False,
                "test_used": False,
            },

            "schema": {
                "generative_columns": generative_columns,
                "feature_columns": feature_columns,
                "numeric_columns": numeric_columns,
                "categorical_columns": categorical_columns,
                "target_column": target,
                "provenance_column": provenance,
                "identifier_columns": identifiers,
            },

            "leakage_policy": {
                "target_used_as_predictor": False,
                "provenance_used": False,
                "identifiers_used": False,
            },

            "sampling": {
                "policy": SYNTHETIC_SAMPLE_POLICY,
                "training_rows": TRAINING_ROWS[
                    dataset_id
                ],
                "synthetic_rows": len(
                    SYNTHETIC_DATA[
                        dataset_id
                    ][baseline_name]
                ),
            },

            "reproducibility": {
                "master_seed": MASTER_SEED,
                "baseline_seed": seed,
            },

            "runtime": RUNTIME_DF[
                (
                    RUNTIME_DF["dataset_id"]
                    == dataset_id
                )
                &
                (
                    RUNTIME_DF["baseline"]
                    == baseline_name
                )
            ].to_dict(
                orient="records"
            ),

            "created_utc": datetime.now(
                timezone.utc
            ).isoformat(),
        }

        metadata_path = (
            NB04_METADATA_ROOT
            / dataset_id
            / f"{baseline_name}.json"
        )

        metadata_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        with open(
            metadata_path,
            "w",
            encoding="utf-8",
        ) as handle:

            json.dump(
                metadata,
                handle,
                indent=2,
                default=str,
            )

        BASELINE_METADATA_RECORDS.append(
            {
                "dataset_id": dataset_id,
                "baseline": baseline_name,
                "metadata_path": str(
                    metadata_path
                ),
                "metadata_sha256": calculate_sha256(
                    metadata_path
                ),
            }
        )

        print(
            f"✓ {dataset_id:<20} | "
            f"{baseline_name:<24} | "
            f"metadata saved"
        )

# Save runtime table
RUNTIME_PATH = (
    NB04_METADATA_ROOT
    / "baseline_runtime.csv"
)

RUNTIME_DF.to_csv(
    RUNTIME_PATH,
    index=False,
)

BASELINE_METADATA_DF = pd.DataFrame(
    BASELINE_METADATA_RECORDS
)

print()
print("✓ Baseline metadata saved.")
print(
    f"✓ Runtime table saved : {RUNTIME_PATH}"
)

SECTION 14 — SAVE BASELINE METADATA

✓ SECTION 14 PASS


In [26]:
# ==================================================================================================
# 15. SAVE GENERATION MANIFEST
# ==================================================================================================

print("=" * 100)
print("SECTION 15 — SAVE GENERATION MANIFEST")
print("=" * 100)

import joblib

GENERATION_MANIFEST_RECORDS = []

for dataset_id in DATASET_IDS:

    for baseline_name in BASELINE_METHODS:

        # ------------------------------------------------------------------------------------------
        # Model path
        # ------------------------------------------------------------------------------------------

        model_directory = (
            NB04_MODEL_ROOT
            / dataset_id
        )

        model_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        model_path = (
            model_directory
            / f"{baseline_name}.pkl"
        )

        # ------------------------------------------------------------------------------------------
        # Save fitted model
        # ------------------------------------------------------------------------------------------

        fitted_model = (
            FITTED_BASELINE_MODELS[
                dataset_id
            ][baseline_name]
        )

        if baseline_name == "independent_marginal":

            joblib.dump(
                fitted_model,
                model_path,
                compress=3,
            )

        elif baseline_name == "gaussian_copula":

            fitted_model[
                "synthesizer"
            ].save(
                filepath=str(
                    model_path
                )
            )

        else:

            raise ValueError(
                f"Unknown baseline: {baseline_name}"
            )

        # ------------------------------------------------------------------------------------------
        # Synthetic artifact
        # ------------------------------------------------------------------------------------------

        synthetic_record = (
            SYNTHETIC_ARTIFACT_DF[
                (
                    SYNTHETIC_ARTIFACT_DF[
                        "dataset_id"
                    ]
                    == dataset_id
                )
                &
                (
                    SYNTHETIC_ARTIFACT_DF[
                        "baseline"
                    ]
                    == baseline_name
                )
            ]
            .iloc[0]
        )

        # ------------------------------------------------------------------------------------------
        # Metadata artifact
        # ------------------------------------------------------------------------------------------

        metadata_record = (
            BASELINE_METADATA_DF[
                (
                    BASELINE_METADATA_DF[
                        "dataset_id"
                    ]
                    == dataset_id
                )
                &
                (
                    BASELINE_METADATA_DF[
                        "baseline"
                    ]
                    == baseline_name
                )
            ]
            .iloc[0]
        )

        runtime_record = (
            RUNTIME_DF[
                (
                    RUNTIME_DF[
                        "dataset_id"
                    ]
                    == dataset_id
                )
                &
                (
                    RUNTIME_DF[
                        "baseline"
                    ]
                    == baseline_name
                )
            ]
            .iloc[0]
        )

        # ------------------------------------------------------------------------------------------
        # Manifest record
        # ------------------------------------------------------------------------------------------

        GENERATION_MANIFEST_RECORDS.append(
            {

                "notebook_id": NOTEBOOK_ID,

                "notebook_version": NOTEBOOK_VERSION,

                "dataset_id": dataset_id,

                "baseline": baseline_name,

                "fit_split": "train",

                "validation_used_for_fit": False,

                "test_used_for_fit": False,

                "target_used_as_predictor": False,

                "identifier_used": False,

                "provenance_used": False,

                "training_rows": TRAINING_ROWS[
                    dataset_id
                ],

                "synthetic_rows": int(
                    synthetic_record["rows"]
                ),

                "synthetic_columns": int(
                    synthetic_record["columns"]
                ),

                "synthetic_data_path": (
                    synthetic_record[
                        "relative_path"
                    ]
                ),

                "synthetic_data_sha256": (
                    synthetic_record[
                        "sha256"
                    ]
                ),

                "synthetic_file_size_bytes": (
                    synthetic_record[
                        "file_size_bytes"
                    ]
                ),

                "model_path": str(
                    model_path.relative_to(
                        NB04_ROOT
                    )
                ),

                "model_sha256": calculate_sha256(
                    model_path
                ),

                "metadata_path": str(
                    Path(
                        metadata_record[
                            "metadata_path"
                        ]
                    ).relative_to(
                        NB04_ROOT
                    )
                ),

                "metadata_sha256": (
                    metadata_record[
                        "metadata_sha256"
                    ]
                ),

                "seed": int(
                    runtime_record[
                        "seed"
                    ]
                ),

                "fit_runtime_seconds": float(
                    runtime_record[
                        "fit_runtime_seconds"
                    ]
                ),

                "generation_runtime_seconds": float(
                    runtime_record[
                        "generation_runtime_seconds"
                    ]
                ),

                "total_runtime_seconds": float(
                    runtime_record[
                        "total_runtime_seconds"
                    ]
                ),

                "status": "PASS",

                "created_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
            }
        )

GENERATION_MANIFEST_DF = pd.DataFrame(
    GENERATION_MANIFEST_RECORDS
)

GENERATION_MANIFEST_PATH = (
    NB04_MANIFEST_ROOT
    / "generation_manifest.csv"
)

GENERATION_MANIFEST_DF.to_csv(
    GENERATION_MANIFEST_PATH,
    index=False,
)

print(
    f"✓ Generation manifest saved:\n"
    f"  {GENERATION_MANIFEST_PATH}"
)

print(
    f"✓ Manifest records : "
    f"{len(GENERATION_MANIFEST_DF)}"
)

SECTION 15 — SAVE GENERATION MANIFEST


NameError: name 'RUNTIME_DF' is not defined

In [28]:
# ==================================================================================================
# 16. VERIFY ARTIFACTS
# ==================================================================================================

print("=" * 100)
print("SECTION 16 — VERIFY ARTIFACTS")
print("=" * 100)

EXPECTED_RUNS = (
    len(DATASET_IDS)
    *
    len(BASELINE_METHODS)
)

if len(GENERATION_MANIFEST_DF) != EXPECTED_RUNS:

    raise RuntimeError(
        "Generation manifest count mismatch.\n"
        f"Expected: {EXPECTED_RUNS}\n"
        f"Found   : {len(GENERATION_MANIFEST_DF)}"
    )

for _, record in GENERATION_MANIFEST_DF.iterrows():

    dataset_id = record[
        "dataset_id"
    ]

    baseline_name = record[
        "baseline"
    ]

    # ----------------------------------------------------------------------------------------------
    # Paths
    # ----------------------------------------------------------------------------------------------

    synthetic_path = (
        NB04_ROOT
        / record[
            "synthetic_data_path"
        ]
    )

    model_path = (
        NB04_ROOT
        / record[
            "model_path"
        ]
    )

    metadata_path = (
        NB04_ROOT
        / record[
            "metadata_path"
        ]
    )

    # ----------------------------------------------------------------------------------------------
    # Existence
    # ----------------------------------------------------------------------------------------------

    for label, path in [
        ("synthetic data", synthetic_path),
        ("model", model_path),
        ("metadata", metadata_path),
    ]:

        if not path.exists():

            raise FileNotFoundError(
                f"{dataset_id}/{baseline_name}: "
                f"{label} artifact missing:\n{path}"
            )

    # ----------------------------------------------------------------------------------------------
    # Synthetic hash
    # ----------------------------------------------------------------------------------------------

    actual_synthetic_hash = calculate_sha256(
        synthetic_path
    )

    if actual_synthetic_hash != record[
        "synthetic_data_sha256"
    ]:

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "synthetic-data SHA-256 mismatch."
        )

    # ----------------------------------------------------------------------------------------------
    # Model hash
    # ----------------------------------------------------------------------------------------------

    actual_model_hash = calculate_sha256(
        model_path
    )

    if actual_model_hash != record[
        "model_sha256"
    ]:

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "model SHA-256 mismatch."
        )

    # ----------------------------------------------------------------------------------------------
    # Metadata hash
    # ----------------------------------------------------------------------------------------------

    actual_metadata_hash = calculate_sha256(
        metadata_path
    )

    if actual_metadata_hash != record[
        "metadata_sha256"
    ]:

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "metadata SHA-256 mismatch."
        )

    # ----------------------------------------------------------------------------------------------
    # Reload persisted synthetic dataset
    # ----------------------------------------------------------------------------------------------

    reloaded_df = pd.read_csv(
        synthetic_path,
        low_memory=False,
    )

    expected_columns = (
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    if list(
        reloaded_df.columns
    ) != list(
        expected_columns
    ):

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "persisted synthetic schema mismatch."
        )

    expected_rows = TRAINING_ROWS[
        dataset_id
    ]

    if len(reloaded_df) != expected_rows:

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "persisted synthetic row count mismatch."
        )

    # ----------------------------------------------------------------------------------------------
    # Verify forbidden columns
    # ----------------------------------------------------------------------------------------------

    provenance = (
        TRAINING_PROVENANCE_COLUMNS[
            dataset_id
        ]
    )

    identifiers = (
        TRAINING_IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )

    if provenance in reloaded_df.columns:

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            "persisted provenance leakage."
        )

    leaked_ids = (
        set(identifiers)
        .intersection(
            reloaded_df.columns
        )
    )

    if leaked_ids:

        raise RuntimeError(
            f"{dataset_id}/{baseline_name}: "
            f"persisted identifier leakage: {leaked_ids}"
        )

    print(
        f"✓ {dataset_id:<20} | "
        f"{baseline_name:<24} | "
        f"CSV PASS | MODEL PASS | METADATA PASS"
    )

# --------------------------------------------------------------------------------------------------
# Save artifact validation report
# --------------------------------------------------------------------------------------------------

ARTIFACT_VALIDATION_REPORT = {

    "notebook_id": NOTEBOOK_ID,

    "notebook_version": NOTEBOOK_VERSION,

    "expected_runs": EXPECTED_RUNS,

    "verified_runs": EXPECTED_RUNS,

    "datasets": DATASET_IDS,

    "baselines": BASELINE_METHODS,

    "all_synthetic_data_verified": True,

    "all_models_verified": True,

    "all_metadata_verified": True,

    "schema_verification": True,

    "sample_size_verification": True,

    "target_separation_verification": True,

    "identifier_exclusion_verification": True,

    "provenance_exclusion_verification": True,

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

ARTIFACT_VALIDATION_PATH = (
    NB04_VALIDATION_ROOT
    / "artifact_validation.json"
)

with open(
    ARTIFACT_VALIDATION_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        ARTIFACT_VALIDATION_REPORT,
        handle,
        indent=2,
    )

print()
print(
    f"✓ Artifact validation report saved:\n"
    f"  {ARTIFACT_VALIDATION_PATH}"
)

print()
print("✓ SECTION 16 — ARTIFACT VERIFICATION : PASS")

SECTION 16 — VERIFY ARTIFACTS


NameError: name 'DATASET_IDS' is not defined

In [29]:
# ==================================================================================================
# 17. COMPLETION SUMMARY
# ==================================================================================================

print("=" * 100)
print("SECTION 17 — COMPLETION SUMMARY")
print("=" * 100)

print()
print("NOTEBOOK 04 — STATISTICAL BASELINES")
print("=" * 100)

print(
    f"Notebook version       : {NOTEBOOK_VERSION}"
)

print(
    f"Project root           : {PROJECT_ROOT}"
)

print(
    f"Notebook 02 input      : {CANONICAL_NB02_ROOT}"
)

print(
    f"Notebook 04 output     : {NB04_ROOT}"
)

print()
print("DATASETS")
print("-" * 100)

for dataset_id in DATASET_IDS:

    print(
        f"✓ {dataset_id:<20} | "
        f"training rows={TRAINING_ROWS[dataset_id]:,}"
    )

print()
print("BASELINES")
print("-" * 100)

for baseline_name in BASELINE_METHODS:

    print(
        f"✓ {baseline_name:<24} | "
        f"{BASELINE_DEFINITIONS[baseline_name]['name']}"
    )

print()
print("EXPERIMENTAL INTEGRITY")
print("-" * 100)

print("✓ TRAIN split only used for fitting")
print("✓ VALIDATION split excluded from fitting")
print("✓ TEST split excluded from fitting")
print("✓ Notebook 02 preprocessing not refitted")
print("✓ Raw datasets not reloaded")
print("✓ Native generative schema used")
print("✓ Target retained in synthetic data")
print("✓ Target never used as predictor")
print("✓ Provenance excluded")
print("✓ Identifiers excluded")
print("✓ Synthetic size equals training size")
print("✓ Reproducible seed policy applied")

print()
print("ARTIFACTS")
print("-" * 100)

print(
    f"✓ Synthetic datasets : "
    f"{len(SYNTHETIC_ARTIFACT_DF)}"
)

print(
    f"✓ Model artifacts    : "
    f"{EXPECTED_RUNS}"
)

print(
    f"✓ Metadata artifacts : "
    f"{EXPECTED_RUNS}"
)

print(
    f"✓ Manifest records   : "
    f"{len(GENERATION_MANIFEST_DF)}"
)

print()
print("OUTPUT LOCATIONS")
print("-" * 100)

print(
    f"Synthetic : {NB04_SYNTHETIC_ROOT}"
)

print(
    f"Models    : {NB04_MODEL_ROOT}"
)

print(
    f"Metadata  : {NB04_METADATA_ROOT}"
)

print(
    f"Manifests : {NB04_MANIFEST_ROOT}"
)

print(
    f"Validation: {NB04_VALIDATION_ROOT}"
)

print()
print("=" * 100)
print("✓ NOTEBOOK 04 — STATISTICAL BASELINES : PASS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# RAM cleanup of temporary fitting objects
# --------------------------------------------------------------------------------------------------

gc.collect()

print()
print("✓ RAM cleanup completed.")

SECTION 16 — VERIFY ARTIFACTS


NameError: name 'DATASET_IDS' is not defined